### Group 24

Shiref Khaled Elhalawany -  221100944

Ahmed Anis Hassan - 221100101 

Karim Ashraf Elsayed - 221100391

Kareem Shaheen - 221101524

# Part 1: Domain Analysis and Data Preparation (4%)

## 1. Domain Background and Requirements

### 1.1. Domain Background

**Domain Definition & Problem Overview**  
The domain focuses on **Open Dataset Recommendation for Researchers**, aiming to assist academics and data scientists in discovering relevant datasets and resources (represented here by books). Researchers often face information overload, struggling to identify high-quality, relevant resources amidst vast, unorganized repositories. Current recommendation approaches in this domain typically rely on simple keyword searches, popularity metrics, or manual curation, which fail to capture the specific, evolving research interests of individual users or the semantic context of the resources.

**Proposed System Focus & Target Users**  
Our proposed system implements a **Hybrid Recommender System** that integrates Content-Based Filtering with Collaborative Filtering. By analyzing the semantic content of resource descriptions and the behavioral patterns of users, the system aims to provide personalized, context-aware recommendations. The target users are **researchers, students, and data professionals** seeking to optimize their literature review and data discovery processes. This system is useful because it bridges the gap between query-based search and serendipitous discovery, reducing research time and uncovering valuable but less obvious resources.

### 1.2. Key Domain Challenges

**1. Cold-Start Scenarios**  
A major challenge is the cold-start problem, where new researchers join the platform without any prior interaction history, or new datasets are added with no ratings. In these cases, pure collaborative filtering fails. Our system must leverage content attributes (titles, descriptions, categories) to provide meaningful initial recommendations until sufficient interaction data is accrued.

**2. Data Sparsity**  
In academic domains, the interaction matrix is typically highly sparse; a single researcher engages with a minuscule fraction of the total available resources. This sparsity makes it difficult to compute reliable user similarities or neighbor variances. We will address this by employing dimensionality reduction techniques (like SVD) and hybrid strategies to infer preferences even with limited data points.

**3. Scalability Considerations**  
Real-time requirements pose a challenge as the number of users and items grows. Computing pairwise similarities (e.g., for k-NN) can become prohibitively expensive ($O(N^2)$). The system requires efficient implementation of algorithms and potentially model-based approaches to ensure that top-N recommendations can be generated quickly without latency issues during user interaction.

## 2. Dataset Preparation

### 2.1. Data Source

**Dataset Name**: Amazon Reviews 2023  
**Subset**: Books (5-core)  
**Source**: [Amazon Reviews 2023](https://amazon-reviews-2023.github.io/data_processing/5core.html)  

We use the Amazon Reviews 2023 dataset, specifically the Books 5-core subset. This version guarantees that all users and items have at least 5 reviews, ensuring a dense enough core for meaningful collaborative filtering analysis.

**Files Required:**
- **Interactions**: `Books.csv` (Contains user ratings, timestamps, and item IDs)
- **Metadata**: `meta_Books.jsonl` (Contains detailed item attributes like title, description, categories, price)

**Dataset Content:**
The dataset includes:
- **User Behavior**: `user_id`, `parent_asin` (Item ID), `rating`, `timestamp`.
- **Content Features**: `title`, `description` (text), `categories` (hierarchical layout), `price`, `average_rating`.

### 2.2. Data Verification

In [ ]:
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math
import re
from collections import Counter

DATA_DIR = "../data"
BOOKS_CSV_PATH = os.path.join(DATA_DIR, "Books.csv")
META_JSONL_PATH = os.path.join(DATA_DIR, "meta_Books.jsonl")
RESULTS_DIR = "../results"
if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR)

def check_dataset_requirements():
    print("Checking Dataset Requirements...\n")
    
    print(f"Loading {BOOKS_CSV_PATH}...")
    if not os.path.exists(BOOKS_CSV_PATH):
        print("Error: Books.csv not found.")
        return
        
    df_interactions = pd.read_csv(BOOKS_CSV_PATH)
    
    num_users = df_interactions['user_id'].nunique()
    num_items = df_interactions['parent_asin'].nunique()
    num_interactions = len(df_interactions)
    
    print(f"\n--- Dataset Statistics ---")
    print(f"Number of Users: {num_users:,}")
    print(f"Number of Items: {num_items:,}")
    print(f"Number of Interactions: {num_interactions:,}")
    
    print(f"\n--- Requirement Check ---")
    if num_users >= 5000 and num_items >= 500 and num_interactions >= 50000:
        print("✅ Minimum requirements met (>5k users, >500 items, >50k interactions).")
    else:
        print("❌ Minimum requirements NOT met.")
        
    print(f"\n--- Content Feature Availability ---")
    if os.path.exists(META_JSONL_PATH):
        with open(META_JSONL_PATH, 'r') as f:
            first_line = json.loads(f.readline())
            print(f"Sample Metadata Keys: {list(first_line.keys())}")
            
            key_features = ['title', 'categories', 'description', 'price']
            present_features = [k for k in key_features if k in first_line]
            print(f"Key Features Found: {present_features}")
    else:
        print("Warning: meta_Books.jsonl not found. Content features might be missing.")

### 2.3. Data Preprocessing Pipeline

In [ ]:
def load_data(file_path, file_type='csv', columns=None):
    print(f"Loading data from {file_path}...")
    if file_type == 'csv':
        if columns:
            return pd.read_csv(file_path, usecols=columns)
        return pd.read_csv(file_path)
    elif file_type == 'jsonl':
        data = []
        with open(file_path, 'r') as f:
            for line in f:
                json_obj = json.loads(line)
                if columns:
                    filtered_obj = {k: json_obj.get(k) for k in columns}
                    data.append(filtered_obj)
                else:
                    data.append(json_obj)
        return pd.DataFrame(data)
    else:
        raise ValueError("Unsupported file type. Use 'csv' or 'jsonl'.")

def preprocess_interactions(df):
    print("Preprocessing interactions...")
    df = df[['user_id', 'parent_asin', 'rating']].copy()
    df.columns = ['user_id', 'item_id', 'rating']
    print(f"Selected columns: {list(df.columns)}, Shape: {df.shape}")
    return df

def handle_missing_values(df):
    print("Handling missing values...")
    original_len = len(df)
    df_clean = df.dropna(subset=['user_id', 'item_id', 'rating'])
    dropped = original_len - len(df_clean)
    if dropped > 0:
        print(f"Dropped {dropped} rows with missing values.")
    print(f"Validation: Missing values remaining: {df_clean[['user_id', 'item_id', 'rating']].isnull().sum().sum()}, Final shape: {df_clean.shape}")
    return df_clean

def remove_duplicates(df):
    print("Removing duplicates...")
    original_len = len(df)
    df_clean = df.drop_duplicates(subset=['user_id', 'item_id'], keep='last')
    dropped = original_len - len(df_clean)
    if dropped > 0:
        print(f"Dropped {dropped} duplicate interactions.")
    
    duplicates_remaining = df_clean.duplicated(subset=['user_id', 'item_id']).sum()
    print(f"Validation: Duplicates remaining: {duplicates_remaining}, Final shape: {df_clean.shape}")
    return df_clean

def validate_and_scale_ratings(df):
    print("Validating and scaling ratings...")
    df['rating'] = df['rating'].astype(float)
    
    r_min = df['rating'].min()
    r_max = df['rating'].max()
    
    if r_min != 1.0 or r_max != 5.0:
        print(f"Scaling ratings from [{r_min}, {r_max}] to [1, 5]...")
        if r_max > r_min:
            df['rating'] = 1 + (df['rating'] - r_min) * 4 / (r_max - r_min)
        else:
            df['rating'] = 5.0
            
    print(f"Validation: Rating range: [{df['rating'].min()}, {df['rating'].max()}], Type: {df['rating'].dtype}, Final shape: {df.shape}")
    return df

def merge_metadata(df_interactions, df_meta):
    print("Merging with metadata...")
    meta_clean = df_meta.rename(columns={'parent_asin': 'item_id'}).drop_duplicates(subset='item_id')
    
    merged_df = pd.merge(df_interactions, meta_clean, on='item_id', how='left')
    
    missing_meta = merged_df['title'].isnull().sum() if 'title' in merged_df.columns else "N/A"
    print(f"Validation: Merged shape: {merged_df.shape}, Items without metadata: {missing_meta}")
    return merged_df

def compute_statistics(df):
    print("\n--- Dataset Statistics ---")
    n_users = df['user_id'].nunique()
    n_items = df['item_id'].nunique()
    n_ratings = len(df)
    
    print(f"Number of Users: {n_users:,}")
    print(f"Number of Items: {n_items:,}")
    print(f"Number of Ratings: {n_ratings:,}")
    return n_users, n_items, n_ratings

def compute_sparsity(n_users, n_items, n_ratings):
    print("\n--- Sparsity Analysis ---")
    total_possible = n_users * n_items
    sparsity = 1 - (n_ratings / total_possible)
    print(f"Sparsity Level: {sparsity:.6f} ({sparsity*100:.4f}%)")
    return sparsity

def analyze_rating_distribution(df):
    print("\n--- Rating Distribution ---")
    counts = df['rating'].value_counts().sort_index()
    total = len(df)
    for rating, count in counts.items():
        print(f"Rating {rating}: {count:,} ({count/total*100:.1f}%)")

### 2.3. Basic Exploratory Analysis

In [ ]:
import matplotlib.pyplot as plt
def compute_user_activity(df):
    print("Computing user activity...")
    user_activity = df.groupby('user_id').size().reset_index(name='interaction_count')
    user_activity = user_activity.sort_values(by='interaction_count', ascending=False)
    
    save_path = os.path.join(RESULTS_DIR, "user_activity.csv")
    user_activity.to_csv(save_path, index=False)
    print(f"Saved user activity to {save_path}")
    return user_activity

def plot_user_activity(user_activity):
    print("Plotting user activity...")
    plt.figure(figsize=(10, 6))
    plt.hist(user_activity['interaction_count'], bins=50, log=True, color='skyblue', edgecolor='black')
    plt.title('Distribution of Number of Ratings per User')
    plt.xlabel('Number of Ratings (Log Scale)')
    plt.ylabel('Number of Users (Log Scale)')
    plt.grid(True, which="both", ls="-", alpha=0.5)
    
    save_path = os.path.join(RESULTS_DIR, "user_activity_distribution.png")
    plt.savefig(save_path)
    plt.show()
    print(f"Saved user activity plot to {save_path}")

def compute_item_popularity(df):
    print("Computing item popularity...")
    item_popularity = df.groupby('item_id').size().reset_index(name='interaction_count')
    item_popularity = item_popularity.sort_values(by='interaction_count', ascending=False)
    
    save_path = os.path.join(RESULTS_DIR, "item_popularity.csv")
    item_popularity.to_csv(save_path, index=False)
    print(f"Saved item popularity to {save_path}")
    return item_popularity

def plot_item_popularity(item_popularity):
    print("Plotting item popularity...")
    plt.figure(figsize=(10, 6))
    plt.hist(item_popularity['interaction_count'], bins=50, log=True, color='salmon', edgecolor='black')
    plt.title('Distribution of Number of Ratings per Item')
    plt.xlabel('Number of Ratings (Log Scale)')
    plt.ylabel('Number of Items (Log Scale)')
    plt.grid(True, which="both", ls="-", alpha=0.5)
    
    save_path = os.path.join(RESULTS_DIR, "item_popularity_distribution.png")
    plt.savefig(save_path)
    plt.show()
    print(f"Saved item popularity plot to {save_path}")

def analyze_long_tail(item_popularity):
    print("Analyzing long-tail problem...")
    total_interactions = item_popularity['interaction_count'].sum()
    top_20_percent_idx = int(len(item_popularity) * 0.2)
    
    top_20_interactions = item_popularity.iloc[:top_20_percent_idx]['interaction_count'].sum()
    bottom_80_interactions = total_interactions - top_20_interactions
    
    ratio = top_20_interactions / total_interactions
    
    print(f"Total Interactions: {total_interactions:,}")
    print(f"Top 20% Items Interactions: {top_20_interactions:,} ({ratio*100:.2f}% of total)")
    print(f"Bottom 80% Items Interactions: {bottom_80_interactions:,} ({(1-ratio)*100:.2f}% of total)")
    
    plt.figure(figsize=(10, 6))
    item_count_array = item_popularity['interaction_count'].reset_index(drop=True).values
    plt.plot(item_count_array, color='blue')
    plt.fill_between(range(len(item_count_array)), item_count_array, color='blue', alpha=0.3)
    plt.axvline(x=top_20_percent_idx, color='red', linestyle='--', label='Top 20% Cutoff')
    plt.title('Long Tail Distribution of Item Popularity')
    plt.xlabel('Item Rank')
    plt.ylabel('Number of Ratings')
    plt.legend()
    plt.grid(True)

    save_path = os.path.join(RESULTS_DIR, "long_tail_plot.png")
    plt.savefig(save_path)
    plt.show()
    print(f"Saved long-tail analysis plot to {save_path}")
    
    sorted_df = item_popularity.reset_index(drop=True)
    return ratio, sorted_df

def interpret_results(ratio):
    print("\n--- Interpretation ---")
    if ratio > 0.6:
        print(f"A significant Long-Tail problem exists. The top 20% of items account for {ratio*100:.2f}% of all interactions.")
        print("This indicates that a small number of popular items dominate the user attention (Popularity Bias).")
        print("Recommender system should aim to improve coverage of the less popular items (the 'tail').")
    else:
        print(f"The distribution is more balanced. Top 20% items account for {ratio*100:.2f}% of interactions.")